In [1]:
!pip install mlflow feast evidently fastapi uvicorn scikit-learn xgboost lightgbm

  Using cached typeguard-4.6.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached gunicorn-26.2.0-py3-none-any.whl.metadata (5.5 kB)
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.0 MB 657.4 kB/s eta 0:00:18
   - -------------------------------------- 0.5/12.0 MB 657.4 kB/s eta 0:00:18
   -- ------------------------------------- 0.8/12.0 MB 666.8 kB/s eta 0:00:17
   --- ------------------------------------ 1.0/12.0 MB 695.9 kB/s eta 0:00:16
   --- ------------------------------------ 1.0/12.0 MB 695.9 kB/s eta 0:00:16
   --- ------------------------------------ 1.0/12.0 MB 695.9 kB/s eta 0:00:16
   ---- ----------------------------------- 1.3/12.0 MB 656.6 kB/s eta 0:00:17
   ----- --------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow-task-sdk 1.3.2 requires apache-airflow-core<3.4.0,>=3.3.0, which is not installed.


# Лабораторная работа 4: Интеграция ML-моделей в production

## Введение
- **Датасет:** MovieLens (оценки фильмов)
- **Сценарий:** Рекомендательная система (предсказание высокой оценки)
- **Уникальное требование (цифра 8):** Обработка временных паттернов (сезонность).

## Часть 1. Прототипирование и выбор модели

### 1.1. Подготовка данных (EDA)
Задача: бинарная классификация. Предсказываем, поставит ли пользователь 
высокую оценку (>= 4.0). 
В качестве фичей возьмем:
- `userId`, `movieId` (базовые ID)
- Временные фичи (реализация уникального требования): час, день недели, месяц.

In [2]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Загружаем данные
DATA_PATH = r"C:\Users\Сергей\Desktop\labs\LabsBigData\lab2\work\ratings_100mb.parquet" 
# (Укажи свой путь к файлу ratings_100mb.parquet, который мы создали во 2 лабе)
try:
    df = pd.read_parquet(DATA_PATH)
except:
    # Если пути нет, берем из 3 лабы (если там остался)
    df = pd.read_parquet("work/ratings_100mb.parquet")

# Берем 1 миллион строк для скорости
df = df.sample(n=1_000_000, random_state=42).copy()

# Создаем целевую переменную: 1, если рейтинг >= 4.0, иначе 0
df['target'] = (df['rating'] >= 4.0).astype(int)

# === РЕАЛИЗАЦИЯ УНИКАЛЬНОГО ТРЕБОВАНИЯ (Временные паттерны) ===
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month

# Выбираем фичи для обучения
features = ['userId', 'movieId', 'hour', 'day_of_week', 'month']
X = df[features]
y = df['target']

# Разделяем на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Обучающая выборка: {X_train.shape[0]:,} строк")
print(f"Тестовая выборка:  {X_test.shape[0]:,} строк")
print(f"Доля положительного класса: {y.mean():.2%}")

Обучающая выборка: 800,000 строк
Тестовая выборка:  200,000 строк
Доля положительного класса: 49.78%


### 1.2. Обучение моделей и логирование в MLflow

Обучим две модели: Random Forest и XGBoost. 
Все параметры и метрики залогируем в MLflow.

In [3]:
# Создаем локальный эксперимент
mlflow.set_experiment("MovieLens_Recommendation")

def train_and_log_model(model, model_name, params):
    with mlflow.start_run(run_name=model_name):
        # Логируем параметры
        mlflow.log_params(params)
        
        # Обучение
        model.fit(X_train, y_train)
        
        # Предсказание
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)
        
        # Метрики
        auc = roc_auc_score(y_test, y_pred_proba)
        f1 = f1_score(y_test, y_pred)
        
        # Логируем метрики
        mlflow.log_metric("auc", auc)
        mlflow.log_metric("f1", f1)
        
        # Логируем модель
        mlflow.sklearn.log_model(model, "model")
        
        print(f"[{model_name}] AUC: {auc:.4f}, F1: {f1:.4f}")
        return auc, f1

# Модель 1: Random Forest
rf_params = {"n_estimators": 50, "max_depth": 10, "random_state": 42}
rf_model = RandomForestClassifier(**rf_params)
rf_auc, rf_f1 = train_and_log_model(rf_model, "RandomForest", rf_params)

# Модель 2: XGBoost
xgb_params = {"n_estimators": 50, "max_depth": 5, "learning_rate": 0.1, "random_state": 42}
xgb_model = XGBClassifier(**xgb_params, use_label_encoder=False, eval_metric='logloss')
xgb_auc, xgb_f1 = train_and_log_model(xgb_model, "XGBoost", xgb_params)

PermissionError: [WinError 5] Отказано в доступе: 'C:\\Users\\%D0%A1%D0%B5%D1%80%D0%B3%D0%B5%D0%B9'

### Вывод по Части 1

- Данные подготовлены, добавлены временные фичи (час, день недели, месяц) для учета сезонности.
- Обучены две модели: Random Forest и XGBoost.
- Результаты залогированы в MLflow. 
- *(Далее по результатам сравнения выберем лучшую модель).*